# Υπερμοντελοποίηση: Overfitting

Σε αυτό το notebook θα καταλάβουμε πιο προσεχτικά:
* Τι εννοούμε όταν λέμε **ακρίβεια**
* Πώς/γιατί χωρίζουμε τα δεδομένα σε Train και Test -- δηλαδή σε σετ δεδομένων που χρησιμοποιούμε για εκπαίδευση (Training data) και σε σετ δεδομένων που χρησιμοποιούμε για να εκτιμήσουμε την ακρίβεια του εκπαιδευμένου μοντέλου (Testing data).
* Τι θα πεί **υπερμοντελοποίηση** και γιατί πρέπει πάση θυσία να αποφύγουμε αυτό το φαινόμενο στην Μηχανική Μάθηση.


```
Κωνσταντίνος Καραμανής: constantine@utexas.edu
http://users.ece.utexas.edu/~cmcaram/
The University of Texas at Austin
Archimedes/Athena RC
```

## Ξαναχρησιμοποιούμε τα δεδομένα του Διαβήτη

Έχουμε δεί στο προηγούμενο Colab με τον διαβήτη (σε [αυτό το λινκ](https://colab.research.google.com/drive/162hNnEDlmoAzcNFhEMQFmMXFQeVPHgHr?usp=sharing)) πως με ρηχό δέντρο απόφασης (1 ή 2 επίπεδα) πετυχαίνουμε ακρίβεια περίπου 74-77%, ενώ με πολύ πιο βαθύ δέντρο -- π.χ. 10 ή και παραπάνω επίπεδα, πετυχαίνουμε ακρίβεια πάνω από 95%.

Αυτό το συμπέρασμα όμως, είναι παραπλανητικό. **Ποιό λάθος κάνουμε**;

* Το λάθος μας είναι πως εκτιμούμε την ακρίβεια του εκπαιδευμένου αλγορίθμου πάνω στα ίδια δεδομένα που χρησιμοποιήσαμε για την εκπαίδευση!
* **Πρέπει λοιπόν να χωρίσουμε τα δεδομένα σε δύο ομάδες**: δεδομένα που χρησιμοποιούμε για εκπαίδευση -- αυτά λέγονται **Training Data**, -- και δεδομένα που θα χρησιμοποιήσουμε για εκτίμηση ακρίβειας -- αυτά λέγονται **Testing Data**.

In [ ]:
# import important python libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import default_rng
%matplotlib inline

In [ ]:
# sets the seed for NumPy's random number generator to a specific value, in this case, 42
# running code always gets same random numbers -- useful for debugging and verifying results.
np.random.seed(42)

In [ ]:
# μεταφορτώνουμε τα δεδομένα του διαβήτη όπως κάναμε
# και τα καθαρίζουμε

# Mount Google drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Replace 'path_to_your_file.csv' with the actual path to your CSV file
#file_path = '/content/drive/MyDrive/Colab Notebooks/path_to_your_file.csv'

file_path = '/content/drive/MyDrive/Colab Notebooks/YouTube-Data-Sets/diabetes.csv'
diabetes_data = pd.read_csv(file_path) #reads the CSV file located at file_path,
                                       #loads the data into a Pandas DataFrame
                                       #and assigns it to the variable diabetes_data.

In [ ]:
# select rows where the 'Glucose',BMI and BloodPressure values are not equal to 0.
filtered_data = diabetes_data[(diabetes_data['Glucose'] != 0) & (diabetes_data['BMI'] != 0) & (diabetes_data['BloodPressure'] != 0)]
# create a NumPy array containing all the columns from the filtered_data DataFrame except for the 'Outcome' column
X_full = filtered_data.loc[:, filtered_data.columns != 'Outcome'].values #loc is used to access a group of rows and columns by labels
# extract the 'Outcome' column from the filtered_data DataFrame and store it in the variable y
y = filtered_data['Outcome']

### Πόσα δεδομένα μας μένουνε;

In [ ]:
print(X_full.shape)
print(y.shape)

### Χωρίζουμε σε Train και Test

Είναι πιο δύσκολο (απαιτεί περισότερα δεδομένα) να εκπαιδεύσεις έναν αλγόριθμο από το να εκτιμήσεις την ακρίβεια. Οπότε εκπαιδεύουμε με 75% των δεδομένων, και εκτιμούμε ακρίβεια με 25%.

In [ ]:
from sklearn.model_selection import train_test_split #function that splits the dataset into training and testing subsets


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_full, y, test_size=0.25, shuffle=True, random_state=42) #Specifies that 25% of the data should be allocated to the test set
#Indicates that the data should be shuffled before splitting and sets the seed for the random number generator

### Ας εκπαιδεύσουμε ένα βαθύ δέντρο

In [ ]:
#import necessary classes, modules from the scikit-learn library
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn import tree


In [ ]:
d = 10 # depth of the tree

# Create a decision tree
deeptree = DecisionTreeClassifier(max_depth=d, random_state=42)

# Fit the model on the training data
deeptree.fit(X_train, y_train)

### Πόσο καλά τα πήγαμε;

In [ ]:
# Make predictions
y_pred = deeptree.predict(X_train)

# Compute accuracy
accuracy = accuracy_score(y_train, y_pred)
print("Training accuracy of the deep decision tree:", accuracy)

### Χρησιμοποιήσαμε τα δεδομένα εκπαίδευσης

Εκτιμήσαμε την ακρίβεια χρησιμοποιώντας τα δεδομένα εκπαίδευσης (training data).

Αυτή η ακρίβεια στην πρόβλεψη ισχύει ακόμα για δεδομένα που δεν έχει ακόμα δεί το μοντέλο μας κατά την διάρκεια της εκπαίδευσης;

Για να το δούμε αυτό, πρέπει να εκτιμήσουμε την ακρίβεια όχι με τα δεδομένα εκπαίδευσης (training data) αλλά με τα δεδομένα εκτίμησης (testing data)


In [ ]:
# Make predictions
y_pred = deeptree.predict(X_test)

# Compute accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Testing accuracy of the deep decision tree:", accuracy)

### Εκπαιδεύουμε Δέντρα Κάθε Βάθους

Για να μελετήσουμε πιο προσεκτικά το φαινόμενο, εκπαιδεύουμε δέντρα βάθους 1 με 15, και βλέπουμε την ακρίβεια των μοντέλων μας χρησιμοποιώντας τα δύο σετ δεδομένων: Training Data & Testing Data.

**Τι αποτέλεσμα θα περιμέναμε; Πως θα μοιάζουνε οι καμπύλες; Θα ανεβαίνουνε; Και οι δύο; Καμία; Μόνο μία;**

In [ ]:
training_scores = [] #initialize an empty list
testing_scores = [] #initialize an empty list
depth_values = range(15) # create a range object representing integers 0 to 14
for depth in depth_values: #this loop runs 15 times
    dt = tree.DecisionTreeClassifier(max_depth=depth+1) #creates trees of depth 1 to 15
    dt.fit(X_train,y_train)
    train_score = dt.score(X_train,y_train) #calculates the accuracy of the model on the given dataset
    test_score = dt.score(X_test, y_test)
    training_scores.append(train_score) #add train score data to the training_scores list at each iteration
    testing_scores.append(test_score) #add test score data to the testing_scores list at each iteration

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()  # Create a figure containing a single axes.
ax.plot(training_scores,label='training')
ax.plot(testing_scores,label='testing')
ax.set_xlabel('tree depth')  # Add an x-label to the axes.
ax.set_ylabel('accuracy')  # Add a y-label to the axes.
ax.set_title("Train vs Test")  # Add a title to the axes.
ax.legend();  # Add a legend.

### Και το XGBoost

Ας δούμε και το XGBoost -- το μοντέλο που είχαμε χρησιμοποιήσει στο προηγούμενο Notebook.

In [ ]:
from xgboost import XGBClassifier

# Create a Gradient Boosting model
GB_model = XGBClassifier()

# Fit the model on the training data
GB_model.fit(X_train,y_train)

# Compute accuracy
y_pred_gb_train = GB_model.predict(X_train)
y_pred_gb_test = GB_model.predict(X_test)
accuracy_train = accuracy_score(y_train, y_pred_gb_train)
accuracy_test = accuracy_score(y_test, y_pred_gb_test)
print("Training Accuracy of Gradient Boosting:", accuracy_train)
print("Testing Accuracy of Gradient Boosting:", accuracy_test)

# Άλλο Ένα Απλό Παράδειγμα

Φτιάχνουμε άλλο ένα παράδειγμα για να ξαναδούμε την υπερμοντελοποίηση και γιατί είναι απαραίτητο να την αποφύγουμε.

Σε αυτό το παράδειγμα, χρησιμοποιούμε τεχνητά δεδομένα, δηλαδή, δεδομένα που δημιουργούμε εμείς τυχαία.

Παρόλο που ο σκοπός της μηχανικής μάθησης είναι η εύρεση συσχετίσεων σε πραγματικά δεδομένα, είναι χρήσιμο να ελέγχουμε την συμπεριφορά των αλγορίθμων μας σε τεχνητά δεδομένα που ελέγχουμε πλήρως.

### Τα δεδομένα μας

Δημιουργούμε δισδυάστατα δεδομένα, δηλαδή με δύο χαρακτηριστικά εισόδου (features). Οπότε ο πίνακας $X$ θα έχει δύο στήλες, ενώ ο πίνακας $X$ με τα δεδομένα του διαβήτη είχε 8 στήλες για να περιγράψει τους ασθενείς.

Τα δεδομένα είναι τυχαία σημεία στον τετράγωνο [0,10] x [0,10]. Το αποτέλεσμα ("outcome") $y$, υπολογίζεται με τον εξής τρόπο:

1. 95% των σημείων με $X_1 > 5$ έχουν $y = 1$
2. 5% των σημείων με $X_1 > 5$ έχουν $y = 0$
3. 95% των σημείων με $X_1 < 5$ έχουν $y = 0$
4. 5% των σημείων με $X_1 < 5$ έχουν $y = 1$

**Πληροφορία / Θόρυβος**: Ο καλύτερος κανόνας ταξινόμησης είναι ο απλός κανόνας: $X_1 > 5$ ή $X_1 < 5$ (δηλαδή, ένα δέντρο απόφασης βάθους 1), διότι αυτή είναι η μόνη **πληροφορία** που καθορίζει την τιμή του $y$. Υπάρχει όμως και **θόρυβος**: 5% των σημείων, διαλεγμένο τυχαία, είναι ετικετοποιημένα ("labeled") με αντεστραμμένο $y$. Αφού είναι τυχαία διαλεγμένα, *δεν υπάρχει τρόπος να μοντελοποιηθεί αυτή η αντιστροφή* -- είναι ο *θόρυβος* των δεδομένων.

Θα δούμε πως όταν χρησιμοποιούμε βαθύ δέντρο απόφασης, το δέντρο προσπαθεί να βρεί περίπλοκους κανόνες που εξηγούν τα "0" στα δεξιά, και τα "1" στα αριστερά. Με άλλα λόγια, προσπαθεί να μοντελοποιήσει τον τυχαίο θόρυβο.

In [ ]:
# τεχνητά δεδομένα: data generation mechanism

 # return N labeled points
def gen_labeled_points(N,p):
  # First we generate uniformly distributed points
  rng = default_rng(12) # another way to set the random seed
  X = rng.uniform(low=0.0, high=10.0, size=[N,2]) #specifies lower and upper bound of the range as well as the shape of the output array
  y = np.ones(N) # by default set all labels to 1
  # Now we label them
  for i in range(N):
    # generate a p, 1-p coin
    c = 1
    if (rng.uniform(0,1) > 1-p): c = -1.
    if (X[i,0] > 5): y[i] = y[i]*c
    if (X[i,0] < 5): y[i] = -1*y[i]*c
  y = 0.5*(y+1) # make y 0's and 1's
  y = y.astype(int) # return integers, not floats
  return X,y

In [ ]:
# N σημεία, πιθανότητα αλλαγής χρώματος = p
N = 200
p = 0.05
X,y = gen_labeled_points(N,p)

# Create a scatter plot (διάγραμμα διασποράς)
plt.figure(figsize=(10, 6))
for i in range(len(X)):
    if y[i] == 1:
        plt.scatter(X[i, 0], X[i, 1], color='blue', label='y=1' if 'y=1' not in plt.gca().get_legend_handles_labels()[1] else "")
    else:
        plt.scatter(X[i, 0], X[i, 1], color='red', label='y=0' if 'y=0' not in plt.gca().get_legend_handles_labels()[1] else "")

# Adding labels and title
plt.title('Scatter Plot of Synthetic Data')
plt.legend()
plt.show()

### Χωρίζουμε σε δεδομένα εκπαίδευσης και εκτίμησης

**Σημείωση**: Συνήθως θα επιλέγαμε μεγαλύτερο ποσοστό δεδομένων εκπαίδευσης. Εδώ κάνουμε το αντίθετο για να δούμε την συμπεριφορά της υπερμοντελοποίησης.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.75, shuffle=True, random_state=42)

In [ ]:
# Τα δεδομένα εκπαίδευσης

# Create a scatter plot (διάγραμμα διασποράς)
plt.figure(figsize=(10, 6))
for i in range(len(X_train)):
    if y_train[i] == 1:
        plt.scatter(X_train[i, 0], X_train[i, 1], color='blue', label='y=1' if 'y=1' not in plt.gca().get_legend_handles_labels()[1] else "")
    else:
        plt.scatter(X_train[i, 0], X_train[i, 1], color='red', label='y=0' if 'y=0' not in plt.gca().get_legend_handles_labels()[1] else "")

# Adding labels and title
plt.title('Scatter Plot of Training Data')
plt.legend()
plt.show()

In [ ]:
# Τα δεδομένα εκτίμησης

# Create a scatter plot (διάγραμμα διασποράς)
plt.figure(figsize=(10, 6))
for i in range(len(X_test)):
    if y_test[i] == 1:
        plt.scatter(X_test[i, 0], X_test[i, 1], color='blue', label='y=1' if 'y=1' not in plt.gca().get_legend_handles_labels()[1] else "")
    else:
        plt.scatter(X_test[i, 0], X_test[i, 1], color='red', label='y=0' if 'y=0' not in plt.gca().get_legend_handles_labels()[1] else "")

# Adding labels and title
plt.title('Scatter Plot of Testing Data')
plt.legend()
plt.show()

# Ένα βαθύ, ένα ρηχό

Εκπαιδεύουμε ένα Βαθύ και ένα Ρηχό Δέντρο Απόφασης

In [ ]:
dt_deep = tree.DecisionTreeClassifier(max_depth=6)
dt_deep = dt_deep.fit(X_train, y_train)
train_score_deep = dt_deep.score(X_train,y_train)
print('The training accuracy for the deep tree is:', train_score_deep)
dt_shallow = tree.DecisionTreeClassifier(max_depth=1)
dt_shallow = dt_shallow.fit(X_train, y_train)
train_score_shallow = dt_shallow.score(X_train,y_train)
print('The training accuracy for the shallow tree is:', train_score_shallow)

## Να δούμε τις περιοχές που ορίζουν

Πως κατάφερε το βαθύ δέντρο να πετύχει ακρίβεια 100%;

* Το ρηχό δέντρο δεν υπερμοντελοποιεί: πιάνει την πραγματική τάση που υπάρχει στα δεδομένα.
* Το βαθύ δέντρο υπερμοντελοποιεί: καταφέρνει να μοντελοποιήσει και τον τυχαίο "θόρυβο" στα δεδομένα, και έτσι χάνει την πραγματική τάση.

In [ ]:
# Define the grid range based on your data
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

# Create a meshgrid
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                     np.arange(y_min, y_max, 0.1))
# Predict the outcome on the meshgrid
Z = dt_shallow.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot the decision boundary
plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, -Z, alpha=0.8, cmap=plt.cm.coolwarm)

# Plot the training points
scatter = plt.scatter(X_train[:, 0], X_train[:, 1], c=-y_train, cmap=plt.cm.coolwarm, edgecolor='k')
plt.xlabel('X_1')
plt.ylabel('X_2')
plt.title('Shallow Tree Prediction')
plt.legend(handles=scatter.legend_elements()[0], labels=['blue', 'red'])
plt.show()


In [ ]:
# Define the grid range based on your data
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

# Create a meshgrid
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                     np.arange(y_min, y_max, 0.1))
# Predict the outcome on the meshgrid
Z = dt_deep.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot the decision boundary
plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, -Z, alpha=0.8, cmap=plt.cm.coolwarm)

# Plot the training points
scatter = plt.scatter(X_train[:, 0], X_train[:, 1], c=-y_train, cmap=plt.cm.coolwarm, edgecolor='k')
plt.xlabel('X_1')
plt.ylabel('X_2')
plt.title('Deep Tree Prediction')
plt.legend(handles=scatter.legend_elements()[0], labels=['blue', 'red'])
plt.show()


## Ακρίβεια στα δεδομένα εκτίμησης

Η υπερμοντελοποίηση μας κοστίζει: ο τυχαίος "θόρυβος" που (υπερ)μοντελοποιεί το βαθύ δέντρο, είναι, φυσικά, διαφορετικός στα δεδομένα εκτίμησης.

In [ ]:
train_score_deep = dt_deep.score(X_test,y_test)
print('The testing accuracy for the deep tree is:', train_score_deep)
train_score_shallow = dt_shallow.score(X_test,y_test)
print('The testing accuracy for the shallow tree is:', train_score_shallow)

# Που πέφτουν τα δεδομένα εκτίμησης

In [ ]:
# Define the grid range based on your data
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

# Create a meshgrid
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                     np.arange(y_min, y_max, 0.1))
# Predict the outcome on the meshgrid
Z = dt_deep.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot the decision boundary
plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, -Z, alpha=0.8, cmap=plt.cm.coolwarm)

# Plot the training points
scatter = plt.scatter(X_test[:, 0], X_test[:, 1], c=-y_test, cmap=plt.cm.coolwarm, edgecolor='k')
plt.xlabel('X_1')
plt.ylabel('X_2')
plt.title('Deep Tree Prediction: Testing Data')
plt.legend(handles=scatter.legend_elements()[0], labels=['blue', 'red'])
plt.show()


In [ ]:
# Define the grid range based on your data
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

# Create a meshgrid
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                     np.arange(y_min, y_max, 0.1))
# Predict the outcome on the meshgrid
Z = dt_shallow.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot the decision boundary
plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, -Z, alpha=0.8, cmap=plt.cm.coolwarm)

# Plot the training points
scatter = plt.scatter(X_test[:, 0], X_test[:, 1], c=-y_test, cmap=plt.cm.coolwarm, edgecolor='k')
plt.xlabel('X_1')
plt.ylabel('X_2')
plt.title('Shallow Tree Prediction: Testing Data')
plt.legend(handles=scatter.legend_elements()[0], labels=['blue', 'red'])
plt.show()


### **Άσκηση**: random seed

Για να δείτε πως μας εξυπηρετεί το ``random seed``, και πως αλλάζουν τα δεδομένα που δημιουργούμε, μπορείτε να επαναλάβετε τα παραπάνω πειράματα, αλλάζοντας το ``random seed`` είτε στην δημιουργία των σημείων:

```rng = default_rng(12)```

ή όταν τα χωρίσαμε σε δεδομένα εκπαίδευσης και δεδομένα εκτίμησης

```
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.75, shuffle=True, random_state=42)
```

# Τρίτο Παράδειγμα

Βρίσκουμε το είδος της ίριδας από τέσσερεις μετρήσεις: το μάκρος και πλάτος του σεπάλου, και μάκρος και πλάτος των πετάλων.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/3/3b/Iris_%28plant%29.jpg/600px-Iris_%28plant%29.jpg" width=256px/>

In [ ]:
# the load_iris function is used to load the Iris dataset, which is a classic dataset in machine learning and statistics
# contains measurements of iris flowers from three different species
# Iris setosa, Iris versicolor, and Iris virginica.

from sklearn.datasets import load_iris


In [ ]:
# We import the sklearn data set "iris"
X, y = load_iris(return_X_y=True) #directly return the feature matrix X and the target vector y by setting the return_X_y parameter to True


## Πόσα δείγματα έχουμε;

**Άσκηση**: Πόσα δείγματα, και πόσα χαρακτηριστικά (features);

Συμπληρώστε τα ΧΧΧΧΧ

In [ ]:
# Πόσα δείγματα και πόσα χαρακτηριστικά;

ΧΧΧΧΧΧΧ

In [ ]:
# 150 δείγματα. Χρησιμοποιούμε 100 για εκπαίδευση (training), και 50 για εκτίμηση (testing)
X_tr = X[:100]
y_tr = y[:100]
X_test = X[100:]
y_test = y[100:]

**Άσκηση**: εκπαιδεύουμε ένα δέντρο βάθους 2


In [ ]:
# εκπαιδεύουμε ένα δέντρο βάθους 2

# model =
decision_tree = ΧΧΧΧ

# model.fit
ΧΧΧΧ


tree.plot_tree(decision_tree)

## Ακρίβεια στα δεδομένα εκπαίδευσης

In [ ]:
train_score = decision_tree.score(X_tr,y_tr)
print("Train score: %.4f" % train_score)

### ΟΚ καλά τα πήγαμε

Καλά τα πήγαμε στα δεδομένα εκπαίδευσης. Στα δεδομένα εκτίμησης;

In [ ]:
test_score = decision_tree.score(X_test,y_test)
print("Test score: %.4f" % test_score)

<img src="https://upload.wikimedia.org/wikipedia/commons/3/37/Sad-face.png" width=200px/>



# **Άσκηση/Puzzle**

* Γιατί; Τί πήγε στραβά;
* Και πως θα το διορθώσετε;
* Χρησιμοποιήστε κώδικα σαν αυτόν που είδαμε παραπάνω (συγκεκριμένα, το πρόβλημα είναι ο τρόπος με τον οποίον χωρίσαμε τα δεδομένα σε δεδομένα εκπαίδευσης και δεδομένα εκτίμησης).
